# 04. Classification modelling: 7-day vs 14-day vs 30-day snapshots

## Goal
Compare identical model families on the three signup-relative snapshots. The target is the supplied student-level `target_dropout` label. This notebook evaluates **random student-level interpolation**: students are split into train/validation/test sets, and no student can occur in more than one split. It does not establish that the outcome happened after the snapshot because the source data has no dropout timestamp.

## Evaluation design
- Stratified 60/20/20 split by **student**, reused for all cutoffs.
- Preprocessing is fit on training data only; validation selects a threshold/model; test is used once for final reporting.
- Compare a majority baseline, class-weighted logistic regression, and random forest.
- Report precision, recall, F1, PR-AUC, ROC-AUC, confusion matrix, and threshold selected on validation for F1.
- Missing values are imputed inside the pipeline. Identifier, cutoff date, and snapshot length are excluded from features.


In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (precision_score, recall_score, f1_score, average_precision_score,
                             roc_auc_score, confusion_matrix, classification_report)
from IPython.display import display

ROOT = Path("..").resolve()
DATA = ROOT / "data" / "curated"
OUT = ROOT / "outputs" / "metrics"
OUT.mkdir(parents=True, exist_ok=True)


In [2]:
files = {d: pd.read_csv(DATA / f"classification_snapshot_{d}d.csv") for d in (7, 14, 30)}
for d, df in files.items():
    assert len(df) == 500 and df.student_id.is_unique
    assert df.target_dropout.notna().all()
    print(f"{d}-day: rows={len(df)}, dropout rate={df.target_dropout.mean():.3f}")


7-day: rows=500, dropout rate=0.250
14-day: rows=500, dropout rate=0.250
30-day: rows=500, dropout rate=0.250


## Split by student, once, and reuse

A row-wise split is unnecessary for this one-row-per-student dataset, but the split is still explicitly based on `student_id`. The same IDs go to the same partition for all cutoffs, making comparisons fair.

In [3]:
all_ids = files[14].student_id
y_by_id = files[14].set_index("student_id").loc[all_ids, "target_dropout"].astype(int)
train_ids, temp_ids = train_test_split(all_ids, test_size=0.40, random_state=42, stratify=y_by_id)
temp_y = y_by_id.loc[temp_ids]
val_ids, test_ids = train_test_split(temp_ids, test_size=0.50, random_state=42, stratify=temp_y)
split_ids = {"train": set(train_ids), "validation": set(val_ids), "test": set(test_ids)}
assert not (split_ids["train"] & split_ids["validation"] or split_ids["train"] & split_ids["test"] or split_ids["validation"] & split_ids["test"])
print({k: len(v) for k,v in split_ids.items()})


{'train': 300, 'validation': 100, 'test': 100}


In [4]:
def build_pipeline(df, model_name):
    excluded = {"student_id", "target_dropout", "snapshot_days", "snapshot_cutoff"}
    features = [c for c in df.columns if c not in excluded]
    cat = [c for c in features if df[c].dtype == "object" or str(df[c].dtype).startswith("category")]
    num = [c for c in features if c not in cat]
    numeric = Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True)), ("scale", StandardScaler())])
    categorical = Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))])
    prep = ColumnTransformer([("num", numeric, num), ("cat", categorical, cat)])
    clf = LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42) if model_name=="Logistic Regression" else RandomForestClassifier(n_estimators=400, min_samples_leaf=4, class_weight="balanced_subsample", random_state=42, n_jobs=-1)
    return Pipeline([("prep", prep), ("model", clf)]), features

def safe_auc(metric_fn, y, p):
    try: return metric_fn(y, p)
    except ValueError: return np.nan



In [5]:
validation_rows, test_rows = [], []
for days, df in files.items():
    df = df.set_index("student_id").loc[all_ids].reset_index()
    train = df[df.student_id.isin(split_ids["train"])]
    val = df[df.student_id.isin(split_ids["validation"])]
    test = df[df.student_id.isin(split_ids["test"])]
    for model_name in ["Majority baseline", "Logistic Regression", "Random Forest"]:
        if model_name == "Majority baseline":
            model = DummyClassifier(strategy="prior")
            Xtr, Xv, Xt = train[["attempt_count"]], val[["attempt_count"]], test[["attempt_count"]]
        else:
            model, feature_cols = build_pipeline(df, model_name)
            Xtr, Xv, Xt = train[feature_cols], val[feature_cols], test[feature_cols]
        ytr, yv, yt = train.target_dropout.astype(int), val.target_dropout.astype(int), test.target_dropout.astype(int)
        model.fit(Xtr, ytr)
        pv = model.predict_proba(Xv)[:,1]
        # Threshold is selected using validation only; tie-breaking favors the higher threshold.
        candidates = np.unique(np.r_[0.05, np.arange(0.10,0.91,0.025), 0.95])
        scores = [(f1_score(yv, pv >= t, zero_division=0), t) for t in candidates]
        best_f1, threshold = max(scores, key=lambda z:(z[0], z[1]))
        pt = model.predict_proba(Xt)[:,1]
        row = {"snapshot_days":days,"model":model_name,"threshold":threshold,
               "val_f1":best_f1,"test_precision":precision_score(yt,pt>=threshold,zero_division=0),
               "test_recall":recall_score(yt,pt>=threshold,zero_division=0),
               "test_f1":f1_score(yt,pt>=threshold,zero_division=0),
               "test_PR_AUC":safe_auc(average_precision_score,yt,pt),
               "test_ROC_AUC":safe_auc(roc_auc_score,yt,pt),
               "test_TN":int(confusion_matrix(yt,pt>=threshold,labels=[0,1])[0,0]),
               "test_FP":int(confusion_matrix(yt,pt>=threshold,labels=[0,1])[0,1]),
               "test_FN":int(confusion_matrix(yt,pt>=threshold,labels=[0,1])[1,0]),
               "test_TP":int(confusion_matrix(yt,pt>=threshold,labels=[0,1])[1,1])}
        test_rows.append(row)
results = pd.DataFrame(test_rows).sort_values(["snapshot_days","test_f1"], ascending=[True,False])
display(results.round(3))
results.to_csv(OUT / "classification_cutoff_model_comparison.csv", index=False)


,snapshot_days,model,threshold,val_f1,test_precision,test_recall,test_f1,test_PR_AUC,test_ROC_AUC,test_TN,test_FP,test_FN,test_TP
1,7,Logistic Regression,0.475,0.552,0.410,0.64,0.500,0.534,0.719,52,23,9,16
2,7,Random Forest,0.250,0.529,0.304,0.84,0.447,0.485,0.664,27,48,4,21
0,7,Majority baseline,0.250,0.400,0.250,1.00,0.400,0.250,0.500,0,75,0,25
5,14,Random Forest,0.350,0.597,0.390,0.64,0.485,0.539,0.700,50,25,9,16
4,14,Logistic Regression,0.275,0.564,0.309,0.68,0.425,0.620,0.728,37,38,8,17
3,14,Majority baseline,0.250,0.400,0.250,1.00,0.400,0.250,0.500,0,75,0,25
8,30,Random Forest,0.425,0.727,0.630,0.68,0.654,0.687,0.810,65,10,8,17
7,30,Logistic Regression,0.350,0.648,0.500,0.84,0.627,0.751,0.869,54,21,4,21
6,30,Majority baseline,0.250,0.400,0.250,1.00,0.400,0.250,0.500,0,75,0,25


## Interpretation and limitations

- Choose the cutoff based on the operational trade-off: missing at-risk students (false negatives) versus unnecessary outreach (false positives), not F1 alone.
- The same students are evaluated at each cutoff, so the comparison isolates how much extra early-history information helps.
- This is a random student split, often called **interpolation** across the observed student population. It is not a temporal/extrapolation test.
- Since there is no dropout event timestamp, the current label may describe an outcome that occurred before or after a given snapshot. The model is therefore an association/prediction exercise, not yet a validated prospective early-warning system.
- Small test set and only 125 positive labels mean metrics have high uncertainty. Consider repeated stratified group splits or cross-validation after this initial comparison.
